In [20]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.svm import SVR
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import cross_val_score
import optuna
import joblib

from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.ML.Descriptors import MoleculeDescriptors

In [21]:
train_df = pd.read_csv(r"..\..\..\data\002_processed_data\df_combined_hopv_sql.csv")

print(f"   Loaded {len(train_df)} training samples")
print(f"   HOMO samples: {train_df['homo'].notna().sum()}")
print(f"   LUMO samples: {train_df['lumo'].notna().sum()}")

kaggle_df = pd.read_csv(r"..\..\..\data\003_kaggle_competition_data\25261_regression_sample_submission.csv")

print(f"   Loaded {len(kaggle_df)} Kaggle test samples")

descriptor_names = [desc[0] for desc in Descriptors._descList]
calculator = MoleculeDescriptors.MolecularDescriptorCalculator(descriptor_names)
                                                                        #calculate RDKit descriptors for the whole dataset 
def compute_rdkit_descriptors(smiles):
    """Compute RDKit descriptors for a SMILES string"""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return [np.nan] * len(descriptor_names)
    try:
        return list(calculator.CalcDescriptors(mol))
    except:
        return [np.nan] * len(descriptor_names)
                                                                        #calculate bond atom features 
def extract_bond_atom_features(smiles):
   
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return {f: 0 for f in ['n_bonds', 'n_single_bonds', 'n_double_bonds',
                'n_triple_bonds', 'n_aromatic_bonds', 'count_c', 'count_h',
                'count_o', 'count_n', 'count_s', 'count_f', 'count_cl',
                'count_br', 'count_i', 'count_p', 'count_si']}
    
    bonds = mol.GetBonds()
    atoms = mol.GetAtoms()
    atom_symbols = [atom.GetSymbol() for atom in atoms]
    
    return {
        'n_bonds': len(bonds),
        'n_single_bonds': sum(1 for b in bonds if b.GetBondTypeAsDouble() == 1.0),
        'n_double_bonds': sum(1 for b in bonds if b.GetBondTypeAsDouble() == 2.0),
        'n_triple_bonds': sum(1 for b in bonds if b.GetBondTypeAsDouble() == 3.0),
        'n_aromatic_bonds': sum(1 for b in bonds if b.GetBondTypeAsDouble() == 1.5),
        'count_c': atom_symbols.count('C'),
        'count_h': atom_symbols.count('H'),
        'count_o': atom_symbols.count('O'),
        'count_n': atom_symbols.count('N'),
        'count_s': atom_symbols.count('S'),
        'count_f': atom_symbols.count('F'),
        'count_cl': atom_symbols.count('Cl'),
        'count_br': atom_symbols.count('Br'),
        'count_i': atom_symbols.count('I'),
        'count_p': atom_symbols.count('P'),
        'count_si': atom_symbols.count('Si')
    }


   Loaded 579 training samples
   HOMO samples: 466
   LUMO samples: 454
   Loaded 200 Kaggle test samples


In [22]:
train_descriptors = []
for idx, smiles in enumerate(train_df['smiles']):
    if idx % 100 == 0:
        print(f"     Progress: {idx}/{len(train_df)}...")
    train_descriptors.append(compute_rdkit_descriptors(smiles))

X_train_rdkit = pd.DataFrame(train_descriptors, columns=descriptor_names)

train_bonds = [extract_bond_atom_features(s) for s in train_df['smiles']]
train_bonds_df = pd.DataFrame(train_bonds)

X_train = pd.concat([X_train_rdkit, train_bonds_df], axis=1)
X_train = X_train.loc[:, ~X_train.columns.duplicated()]

# Compute for Kaggle data
kaggle_descriptors = []
for idx, smiles in enumerate(kaggle_df['smiles']):
    if idx % 50 == 0:
        print(f"     Progress: {idx}/{len(kaggle_df)}...")                                  
    kaggle_descriptors.append(compute_rdkit_descriptors(smiles))

X_kaggle_rdkit = pd.DataFrame(kaggle_descriptors, columns=descriptor_names)

kaggle_bonds = [extract_bond_atom_features(s) for s in kaggle_df['smiles']]
kaggle_bonds_df = pd.DataFrame(kaggle_bonds)

X_kaggle = pd.concat([X_kaggle_rdkit, kaggle_bonds_df], axis=1)
X_kaggle = X_kaggle.loc[:, ~X_kaggle.columns.duplicated()]

print(f" Training features: {X_train.shape}")
print(f" Kaggle features: {X_kaggle.shape}")

     Progress: 0/579...
     Progress: 100/579...
     Progress: 200/579...
     Progress: 300/579...
     Progress: 400/579...
     Progress: 500/579...
     Progress: 0/200...
     Progress: 50/200...
     Progress: 100/200...
     Progress: 150/200...
 Training features: (579, 233)
 Kaggle features: (200, 233)


In [23]:
def add_interactions(X):
                                                    #add extra features, only interactions between existing features
    X['TPSA_x_MolLogP'] = X['TPSA'] * X['MolLogP']
    X['aromatic_x_hetero'] = X['NumAromaticRings'] * X['NumHeteroatoms']
    X['bonds_x_carbon'] = X['n_bonds'] * X['count_c']
    X['aromatic_bonds_x_carbon'] = X['n_aromatic_bonds'] * X['count_c']
    X['hetero_to_carbon'] = X['NumHeteroatoms'] / (X['count_c'] + 1)
    X['aromatic_bond_ratio'] = X['n_aromatic_bonds'] / (X['n_bonds'] + 1)
    X['nitrogen_to_carbon'] = X['count_n'] / (X['count_c'] + 1)
    return X

X_train = add_interactions(X_train)
X_kaggle = add_interactions(X_kaggle)
def add_chemistry_features(X):                                                        #add chemistry, knowledge-based features, likely useful
    X['pi_electrons'] = X['n_aromatic_bonds'] * 2 + X['n_double_bonds'] * 2
    X['conjugation_score'] = (X['NumAromaticRings'] * X['n_aromatic_bonds']) / (X['HeavyAtomCount'] + 1)
    X['nitrogen_density'] = X['count_n'] / (X['HeavyAtomCount'] + 1)
    X['oxygen_density'] = X['count_o'] / (X['HeavyAtomCount'] + 1)
    X['hetero_density'] = X['NumHeteroatoms'] / (X['HeavyAtomCount'] + 1)
    X['charge_range'] = X['MaxPartialCharge'] - X['MinPartialCharge']
    X['nitrogen_aromatic'] = X['count_n'] * X['NumAromaticRings']
    X['aromatic_polarity'] = X['NumAromaticRings'] * X['TPSA']
    X['hetero_conjugation'] = X['NumHeteroatoms'] * X['n_aromatic_bonds']
    X['unsaturation'] = (X['n_double_bonds'] + X['n_triple_bonds'] + X['n_aromatic_bonds']) / (X['n_bonds'] + 1)
    X = X.fillna(0).replace([np.inf, -np.inf], 0)
    return X

X_train = add_chemistry_features(X_train)
X_kaggle = add_chemistry_features(X_kaggle)
X_train = X_train.fillna(0).replace([np.inf, -np.inf], 0)
X_kaggle = X_kaggle.fillna(0).replace([np.inf, -np.inf], 0)

# Clip extreme values bcs some values were astronomically high

extreme_clipped = []
for col in X_train.columns:
    if X_train[col].max() > 1e10:
        X_train[col] = X_train[col].clip(upper=1e6)
        extreme_clipped.append(col)

for col in extreme_clipped:
    if col in X_kaggle.columns:
        X_kaggle[col] = X_kaggle[col].clip(upper=1e6)

if extreme_clipped:
    print(f"     Clipped {len(extreme_clipped)} features: {extreme_clipped}")

# Percentile clipping 
'''print("   Percentile clipping...")
for col in X_train.columns:
    if col in extreme_clipped:
        continue
    q99 = X_train[col].quantile(0.99)
    q01 = X_train[col].quantile(0.01)
    if q99 > q01:
        X_train[col] = X_train[col].clip(lower=q01, upper=q99)

# Apply same clipping to Kaggle based on training quantiles
for col in X_kaggle.columns:
    if col in extreme_clipped:
        continue
    if col in X_train.columns:
        q99 = X_train[col].quantile(0.99)
        q01 = X_train[col].quantile(0.01)
        if q99 > q01:
            X_kaggle[col] = X_kaggle[col].clip(lower=q01, upper=q99)    '''

     Clipped 1 features: ['Ipc']


'print("   Percentile clipping...")\nfor col in X_train.columns:\n    if col in extreme_clipped:\n        continue\n    q99 = X_train[col].quantile(0.99)\n    q01 = X_train[col].quantile(0.01)\n    if q99 > q01:\n        X_train[col] = X_train[col].clip(lower=q01, upper=q99)\n\n# Apply same clipping to Kaggle based on training quantiles\nfor col in X_kaggle.columns:\n    if col in extreme_clipped:\n        continue\n    if col in X_train.columns:\n        q99 = X_train[col].quantile(0.99)\n        q01 = X_train[col].quantile(0.01)\n        if q99 > q01:\n            X_kaggle[col] = X_kaggle[col].clip(lower=q01, upper=q99)    '

In [24]:
X_combined = pd.concat([X_train, X_kaggle], axis=0, ignore_index=True)
train_indices = list(range(len(X_train)))
kaggle_indices = list(range(len(X_train), len(X_combined)))

# Scale features for clustering
scaler_cluster = StandardScaler()
X_combined_scaled = scaler_cluster.fit_transform(X_combined)

In [25]:
inertias = []
K_range = range(3, 20)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_combined_scaled)
    inertias.append(kmeans.inertia_)

# Find elbow (simple approach: look for largest decrease)
decreases = [inertias[i] - inertias[i+1] for i in range(len(inertias)-1)]
optimal_k = K_range[np.argmax(decreases)]

print(f"   Optimal clusters: {optimal_k}")

# Perform final clustering
kmeans_final = KMeans(n_clusters=optimal_k, random_state=42, n_init=20)
clusters = kmeans_final.fit_predict(X_combined_scaled)

train_clusters = clusters[train_indices]
kaggle_clusters = clusters[kaggle_indices]

print(f"   Training samples per cluster: {np.bincount(train_clusters)}")
print(f"   Kaggle samples per cluster: {np.bincount(kaggle_clusters)}")

   Optimal clusters: 3
   Training samples per cluster: [310 238  31]
   Kaggle samples per cluster: [200]


In [26]:
# Count how many Kaggle samples are in each cluster
kaggle_cluster_counts = np.bincount(kaggle_clusters, minlength=optimal_k)

# DISTANCE-BASED WEIGHTING 
# Compute distance from each training sample to its cluster center

cluster_centers = kmeans.cluster_centers_
train_distances = np.zeros(len(train_clusters))

# Calculate distances
for i in range(len(train_clusters)):
    cluster_id = train_clusters[i]
    center = cluster_centers[cluster_id]
    # Distance from this training sample to its cluster center
    dist = np.linalg.norm(X_combined_scaled[train_indices[i]] - center)
    train_distances[i] = dist

# Compute weights: (kaggle_count / distance)
# Samples closer to center get higher weight
# Clusters with more Kaggle samples get higher weight
sample_weights_train = np.zeros(len(train_clusters))

for i, cluster_id in enumerate(train_clusters):
    kaggle_count = kaggle_cluster_counts[cluster_id]
    distance = train_distances[i]
    
    # Weight = (kaggle_count + 1) / (distance + 0.1)
    # +1 and +0.1 prevent division by zero
    sample_weights_train[i] = (kaggle_count + 1) / (distance + 0.1)

# Normalize weights to sum to number of samples
sample_weights_train = sample_weights_train / sample_weights_train.sum() * len(sample_weights_train)

print(f"   Distance-based weights range: [{sample_weights_train.min():.2f}, {sample_weights_train.max():.2f}]")
print(f"   Mean weight: {sample_weights_train.mean():.2f}")
print(f"   High weight samples (more than 3): {(sample_weights_train > 3).sum()}")

   Distance-based weights range: [0.01, 2.13]
   Mean weight: 1.00
   High weight samples (more than 3): 0


In [27]:
# HOMO dataset
homo_mask = train_df['homo'].notna()
X_homo = X_train[homo_mask].copy()
y_homo = train_df.loc[homo_mask, 'homo'].values
weights_homo = sample_weights_train[homo_mask]

print(f"   HOMO: {len(X_homo)} samples")

# LUMO dataset  
lumo_mask = train_df['lumo'].notna()
X_lumo = X_train[lumo_mask].copy()
y_lumo = train_df.loc[lumo_mask, 'lumo'].values
weights_lumo = sample_weights_train[lumo_mask]

print(f"   LUMO: {len(X_lumo)} samples")

   HOMO: 466 samples
   LUMO: 454 samples


In [28]:
# SVR requires standardized features!
scaler_homo = StandardScaler()
scaler_lumo = StandardScaler()

X_homo_scaled = scaler_homo.fit_transform(X_homo)
X_lumo_scaled = scaler_lumo.fit_transform(X_lumo)

In [29]:
from sklearn.model_selection import KFold
def objective_homo(trial):
    """Optimize HOMO SVR with manual CV"""
    params = {
        'kernel': trial.suggest_categorical('kernel', ['rbf', 'poly']),
        'C': trial.suggest_float('C', 1.0, 1000.0, log=True),
        'epsilon': trial.suggest_float('epsilon', 0.001, 0.3, log=True),
        'gamma': trial.suggest_categorical('gamma', ['scale', 'auto'])
    }
    
    if params['kernel'] == 'poly':
        params['degree'] = trial.suggest_int('degree', 2, 4)
    
    model = SVR(**params)
    
    # Manual 5-fold CV with sample weights
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = []
    
    for train_idx, val_idx in kf.split(X_homo_scaled):
        X_tr, X_val = X_homo_scaled[train_idx], X_homo_scaled[val_idx]
        y_tr, y_val = y_homo[train_idx], y_homo[val_idx]
        w_tr = weights_homo[train_idx]
        
        # Fit with sample weights
        model.fit(X_tr, y_tr, sample_weight=w_tr)
        
        # Predict validation
        y_pred = model.predict(X_val)
        mae = mean_absolute_error(y_val, y_pred)
        cv_scores.append(-mae)  # Negative for maximization
    
    return np.mean(cv_scores)


optuna.logging.set_verbosity(optuna.logging.WARNING)

study_homo = optuna.create_study(direction='maximize')
study_homo.optimize(objective_homo, n_trials=100, show_progress_bar=True)

print(f"  Best CV MAE: {-study_homo.best_value:.4f}")
print(f"   Best params: {study_homo.best_params}")


Best trial: 93. Best value: -0.118833: 100%|██████████| 100/100 [00:10<00:00,  9.78it/s]

  Best CV MAE: 0.1188
   Best params: {'kernel': 'rbf', 'C': 1.000412659926748, 'epsilon': 0.011403326104113026, 'gamma': 'auto'}


In [30]:
def objective_lumo(trial):
    """Optimize LUMO SVR with manual CV"""
    params = {
        'kernel': trial.suggest_categorical('kernel', ['rbf', 'poly']),
        'C': trial.suggest_float('C', 1.0, 1000.0, log=True),
        'epsilon': trial.suggest_float('epsilon', 0.001, 0.3, log=True),
        'gamma': trial.suggest_categorical('gamma', ['scale', 'auto'])
    }
    
    if params['kernel'] == 'poly':
        params['degree'] = trial.suggest_int('degree', 2, 4)
    
    model = SVR(**params)
    
    # Manual 5-fold CV with sample weights
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = []
    
    for train_idx, val_idx in kf.split(X_lumo_scaled):
        X_tr, X_val = X_lumo_scaled[train_idx], X_lumo_scaled[val_idx]
        y_tr, y_val = y_lumo[train_idx], y_lumo[val_idx]
        w_tr = weights_lumo[train_idx]
        
        model.fit(X_tr, y_tr, sample_weight=w_tr)
        
        y_pred = model.predict(X_val)
        mae = mean_absolute_error(y_val, y_pred)
        cv_scores.append(-mae)
    
    return np.mean(cv_scores)


study_lumo = optuna.create_study(direction='maximize')
study_lumo.optimize(objective_lumo, n_trials=100, show_progress_bar=True)

print(f"   Best CV MAE: {-study_lumo.best_value:.4f}")
print(f"   Best params: {study_lumo.best_params}")

Best trial: 76. Best value: -0.208773: 100%|██████████| 100/100 [00:09<00:00, 10.25it/s]

   Best CV MAE: 0.2088
   Best params: {'kernel': 'rbf', 'C': 1.0041155961275487, 'epsilon': 0.0030263914092687957, 'gamma': 'scale'}


In [31]:
print("   HOMO models:")

#  RBF kernel
svr_homo_rbf = SVR(**study_homo.best_params)
svr_homo_rbf.fit(X_homo_scaled, y_homo, sample_weight=weights_homo)
print(f"     RBF: {len(svr_homo_rbf.support_)} support vectors")

# Polynomial degree=2
params_poly2_homo = {
    'kernel': 'poly', 'degree': 2,
    'C': study_homo.best_params.get('C', 100),
    'epsilon': study_homo.best_params.get('epsilon', 0.01),
    'gamma': 'scale', 'coef0': 1.0
}
svr_homo_poly2 = SVR(**params_poly2_homo)
svr_homo_poly2.fit(X_homo_scaled, y_homo, sample_weight=weights_homo)
print(f"     Poly2: {len(svr_homo_poly2.support_)} support vectors")

# Polynomial degree=3
params_poly3_homo = {
    'kernel': 'poly', 'degree': 3,
    'C': study_homo.best_params.get('C', 100),
    'epsilon': study_homo.best_params.get('epsilon', 0.01),
    'gamma': 'scale', 'coef0': 1.0
}
svr_homo_poly3 = SVR(**params_poly3_homo)
svr_homo_poly3.fit(X_homo_scaled, y_homo, sample_weight=weights_homo)
print(f"     Poly3: {len(svr_homo_poly3.support_)} support vectors")

#lumo models, rbf and poly 2
print("\n   LUMO models:")

#  RBF 
svr_lumo_rbf = SVR(**study_lumo.best_params)
svr_lumo_rbf.fit(X_lumo_scaled, y_lumo, sample_weight=weights_lumo)
print(f"   RBF: {len(svr_lumo_rbf.support_)} support vectors")

# Polynomial degree=2
params_poly2_lumo = {
    'kernel': 'poly', 'degree': 2,
    'C': study_lumo.best_params.get('C', 100),
    'epsilon': study_lumo.best_params.get('epsilon', 0.01),
    'gamma': 'scale', 'coef0': 1.0
}
svr_lumo_poly2 = SVR(**params_poly2_lumo)
svr_lumo_poly2.fit(X_lumo_scaled, y_lumo, sample_weight=weights_lumo)
print(f"     Poly2: {len(svr_lumo_poly2.support_)} support vectors")

print(f"\n  Total: 5 SVR models trained")


   HOMO models:
     RBF: 419 support vectors
     Poly2: 423 support vectors
     Poly3: 425 support vectors

   LUMO models:
   RBF: 446 support vectors
     Poly2: 448 support vectors

  Total: 5 SVR models trained


In [32]:
# HOMO: Predict with all 3 models
homo_pred_rbf_train = svr_homo_rbf.predict(X_homo_scaled)
homo_pred_poly2_train = svr_homo_poly2.predict(X_homo_scaled)
homo_pred_poly3_train = svr_homo_poly3.predict(X_homo_scaled)

# Ensemble average (same weights as Kaggle predictions)
y_pred_homo_train = 0.6 * homo_pred_rbf_train + 0.2 * homo_pred_poly2_train + 0.2 * homo_pred_poly3_train

mae_homo_train = mean_absolute_error(y_homo, y_pred_homo_train)
r2_homo_train = r2_score(y_homo, y_pred_homo_train)

print(f"   HOMO ensemble: MAE = {mae_homo_train:.4f}, R^2 = {r2_homo_train:.4f}")

# LUMO: Predict with 2 models
lumo_pred_rbf_train = svr_lumo_rbf.predict(X_lumo_scaled)
lumo_pred_poly2_train = svr_lumo_poly2.predict(X_lumo_scaled)

y_pred_lumo_train = 0.7 * lumo_pred_rbf_train + 0.3 * lumo_pred_poly2_train

mae_lumo_train = mean_absolute_error(y_lumo, y_pred_lumo_train)
r2_lumo_train = r2_score(y_lumo, y_pred_lumo_train)

print(f"   LUMO ensemble: MAE = {mae_lumo_train:.4f}, R^2 = {r2_lumo_train:.4f}")


   HOMO ensemble: MAE = 0.0807, R^2 = 0.6333
   LUMO ensemble: MAE = 0.1512, R^2 = 0.5726


In [33]:
#Save all 5 SVR ensemble models
joblib.dump(svr_homo_rbf, r'tools_used\regression_svr_homo_rbf.joblib')

joblib.dump(svr_homo_poly2, r'tools_used\regression_svr_homo_poly2.joblib')

joblib.dump(svr_homo_poly3, r'tools_used\regression_svr_homo_poly3.joblib')     
joblib.dump(svr_lumo_rbf, r'tools_used\regression_svr_lumo_rbf.joblib')

joblib.dump(svr_lumo_poly2, r'tools_used\regression_svr_lumo_poly2.joblib')

#Save both scalers
joblib.dump(scaler_homo, r'tools_used\regression_scaler_homo.joblib')

joblib.dump(scaler_lumo, r'tools_used\regression_scaler_lumo.joblib')

#Save feature names
feature_names_homo = X_homo.columns.tolist()
feature_names_lumo = X_lumo.columns.tolist()

joblib.dump(feature_names_homo, r'tools_used\regression_feature_names_homo.joblib')

joblib.dump(feature_names_lumo, r'tools_used\regression_feature_names_lumo.joblib')

joblib.dump(extreme_clipped, r'tools_used\extreme_clipped.joblib')
joblib.dump(kmeans, r'tools_used\kmeans_model.joblib')
joblib.dump(scaler_cluster, r'tools_used\scaler_cluster.joblib')

['tools_used\\scaler_cluster.joblib']

SUBMISSION

In [34]:
import joblib
import numpy as np
import pandas as pd


#Load all 5 SVR ensemble models
svr_homo_rbf = joblib.load(r'tools_used\regression_svr_homo_rbf.joblib')
svr_homo_poly2 = joblib.load(r'tools_used\regression_svr_homo_poly2.joblib')
svr_homo_poly3 = joblib.load(r'tools_used\regression_svr_homo_poly3.joblib')
#lumo
svr_lumo_rbf = joblib.load(r'tools_used\regression_svr_lumo_rbf.joblib')
svr_lumo_poly2 = joblib.load(r'tools_used\regression_svr_lumo_poly2.joblib')
#scalers
scaler_homo = joblib.load(r'tools_used\regression_scaler_homo.joblib')
scaler_lumo = joblib.load(r'tools_used\regression_scaler_lumo.joblib')

# 4. Load feature names
feature_names_homo = joblib.load(r'tools_used\regression_feature_names_homo.joblib')
print(f"Loaded: feature_names_homo ({len(feature_names_homo)} features)")

feature_names_lumo = joblib.load(r'tools_used\regression_feature_names_lumo.joblib')
print(f"Loaded: feature_names_lumo ({len(feature_names_lumo)} features)")





Loaded: feature_names_homo (250 features)
Loaded: feature_names_lumo (250 features)


In [35]:
for feat in X_homo.columns:
    if feat not in X_kaggle.columns:
        X_kaggle[feat] = 0

for feat in X_lumo.columns:
    if feat not in X_kaggle.columns:
        X_kaggle[feat] = 0

X_kaggle_homo = X_kaggle[X_homo.columns].copy()
X_kaggle_lumo = X_kaggle[X_lumo.columns].copy()

# Scale with TRAINING scalers (CRITICAL!)
X_kaggle_homo_scaled = scaler_homo.transform(X_kaggle_homo)
X_kaggle_lumo_scaled = scaler_lumo.transform(X_kaggle_lumo)


In [36]:
# HOMO: Predict with all 3 models
homo_pred_rbf = svr_homo_rbf.predict(X_kaggle_homo_scaled)
homo_pred_poly2 = svr_homo_poly2.predict(X_kaggle_homo_scaled)
homo_pred_poly3 = svr_homo_poly3.predict(X_kaggle_homo_scaled)

# Ensemble average (weighted)
homo_predictions = 0.6 * homo_pred_rbf + 0.2 * homo_pred_poly2 + 0.2 * homo_pred_poly3

print(f"   HOMO ensemble (3 models: RBF 60%, Poly2 20%, Poly3 20%)")
print(f"     Range: [{homo_predictions.min():.4f}, {homo_predictions.max():.4f}]")
print(f"     Mean: {homo_predictions.mean():.4f} ± {homo_predictions.std():.4f}")

# LUMO: Predict with 2 models
lumo_pred_rbf = svr_lumo_rbf.predict(X_kaggle_lumo_scaled)
lumo_pred_poly2 = svr_lumo_poly2.predict(X_kaggle_lumo_scaled)

# Ensemble average
lumo_predictions = 0.7 * lumo_pred_rbf + 0.3 * lumo_pred_poly2

print(f"   LUMO ensemble (2 models: RBF 70%, Poly2 30%)")
print(f"     Range: [{lumo_predictions.min():.4f}, {lumo_predictions.max():.4f}]")
print(f"     Mean: {lumo_predictions.mean():.4f} ± {lumo_predictions.std():.4f}")

   HOMO ensemble (3 models: RBF 60%, Poly2 20%, Poly3 20%)
     Range: [-6.0570, -4.5416]
     Mean: -5.2104 ± 0.2487
   LUMO ensemble (2 models: RBF 70%, Poly2 30%)
     Range: [-4.2444, -2.3306]
     Mean: -3.3516 ± 0.4035


In [37]:
#create the submission df

submission = pd.DataFrame({
    'smiles': kaggle_df['smiles'].values,
    'homo': homo_predictions,
    'lumo': lumo_predictions
})

submission.to_csv(r'tools_used\kaggle_submission_svr_clustering.csv', index=False)

print(f"Saved: kaggle_submission_svr_clustering.csv")
print(f"\n   First 5 predictions:")
print(submission.head().to_string(index=False))


#save models





Saved: kaggle_submission_svr_clustering.csv

   First 5 predictions:
                                                       smiles      homo      lumo
[SiH2]1cccc1-c1ccc(-c2cnc(-c3nccc4nsnc34)c3nsnc23)c2c[nH]cc12 -5.318633 -4.191669
      c1[nH]cc2c1c(cc1c3ccccc3c3c4ccccc4oc3c21)-c1nccc2nsnc12 -5.296329 -3.441656
   c1cc2c(scc2o1)-c1ccc(o1)-c1ccc(-c2cccc3c[nH]cc23)c2ccccc12 -4.963409 -2.916558
       c1cc2csc(-c3cc4c(s3)c3cnc5ccc6ccccc6c5c3c3ccccc43)c2s1 -5.384966 -3.027246
                             C1cccc1-c1ccc(-c2ccccc2)c2nsnc12 -5.281698 -3.248857


In [38]:


print(f"\nClustering:")
print(f"  K = {optimal_k} clusters")
print(f"  Weight strategy: Focus on Kaggle-similar molecules")

print(f"\nModels:")
print(f"  HOMO SVR: {study_homo.best_params}")
print(f"  LUMO SVR: {study_lumo.best_params}")

print(f"\nTraining Performance:")
print(f"  HOMO MAE: {mae_homo_train:.4f}, R²: {r2_homo_train:.4f}")
print(f"  LUMO MAE: {mae_lumo_train:.4f}, R²: {r2_lumo_train:.4f}")

print(f"\nCV Performance:")
print(f"  HOMO CV MAE: {-study_homo.best_value:.4f}")
print(f"  LUMO CV MAE: {-study_lumo.best_value:.4f}")



Clustering:
  K = 3 clusters
  Weight strategy: Focus on Kaggle-similar molecules

Models:
  HOMO SVR: {'kernel': 'rbf', 'C': 1.000412659926748, 'epsilon': 0.011403326104113026, 'gamma': 'auto'}
  LUMO SVR: {'kernel': 'rbf', 'C': 1.0041155961275487, 'epsilon': 0.0030263914092687957, 'gamma': 'scale'}

Training Performance:
  HOMO MAE: 0.0807, R²: 0.6333
  LUMO MAE: 0.1512, R²: 0.5726

CV Performance:
  HOMO CV MAE: 0.1188
  LUMO CV MAE: 0.2088
